In [ ]:
import numpy as np
import matplotlib.pyplot as plt
mu0 = 4 * np.pi * 1e-7
Ms = 1.61 / mu0

In [ ]:
def load_hysteresis_data(filepath):
    """Load hysteresis data from a .npz file."""
    data = np.load(filepath, allow_pickle=True)
    res_tensor = data["res"][1][1,:,:,:]
    
    return {
        'H_array': data["H_array"],
        'M_array': data["M_array"],
        'runtime': float(data["runtime"]),
        'n_points': res_tensor.shape[0]
    }

def plot_hysteresis_comparison(
    series_list,
    ms_value=1.0,
    fig_width_in=6.2,
    fig_height_in=3.8,
    font_size=9,
    label_size=10,
    tick_size=9,
    legend_size=8,
    line_width=1.2,
    marker_size=4.0,
):
    # Update global RC params for poster consistency
    plt.rcParams.update({
        "font.size": font_size,
        "axes.labelsize": label_size,
        "axes.titlesize": label_size,
        "xtick.labelsize": tick_size,
        "ytick.labelsize": tick_size,
        "legend.fontsize": legend_size,
        "font.family": "serif",  # Professional look
    })

    fig, ax = plt.subplots(figsize=(fig_width_in, fig_height_in))

    for s in series_list:
        data = s["data"]
        # Round runtime to nearest integer for the label
        runtime_rounded = int(round(data['runtime']))
        full_label = f"{s['label']} (t={runtime_rounded}s)"
        
        ax.plot(
            data['H_array'], 
            data['M_array'] / ms_value,
            label=full_label,
            color=s.get("color", "black"),
            marker=s.get("marker", None),
            linestyle=s.get("ls", "-"),
            linewidth=line_width,
            markersize=marker_size,
            markevery=0.05, # Avoid cluttering the line with too many markers
            alpha=0.9
        )

    ax.set_xlabel(r"Applied Field, $\mu_0 H$ [ T ]")
    #ax.set_ylabel("Magnetization $M/M_s$ [-]")

    ax.set_ylabel(r"Magnetization, $\mu_0 M$ [ T ]")
    ax.grid(True, which="both", linestyle=":", linewidth=0.8, alpha=0.6)
    
    # Legend layout
    ax.legend(loc="lower right", frameon=True, framealpha=0.9, edgecolor='inherit')
    
    fig.tight_layout(pad=0.2)
    
    # Save as vector graphics
    fig.savefig("hysteresis_comparison.svg", format="svg", bbox_inches="tight")
    fig.savefig("hysteresis_comparison.pdf", format="pdf", bbox_inches="tight")
    
    plt.show()



In [ ]:
# --- Execution ---

base_path = "results_ref/Grid_rasBase_5_nGrains_25_nRef_4_dG_3.75e-09"

# 1. Define configurations (Naming, markers, styles)
# ls examples: '-' (solid), '--' (dashed), (0, (3, 1, 1, 1)) (dash-dot-dot)
configs = [
    {"type": "cuda", "n": None, "l": None, "label": "CUDA",    "marker": None, "ls": "-",  "color": "black"},
    {"type": "cuda_ref",  "n": 6,    "l": 2,    "label": "CUDA_ref",  "marker": "s",  "ls": "--", "color": "tab:blue"},
    #{"type": "fmm",  "n": 4,    "l": 2,    "label": "FMM",  "marker": "s",  "ls": "--", "color": "tab:blue"},
    {"type": "fmm",  "n": 6,    "l": 2,    "label": "N6 L2",  "marker": "s",  "ls": "--", "color": "tab:blue"},
   #{"type": "fmm",  "n": 6,    "l": 3,    "label": "N6 L3",  "marker": "o",  "ls": "--", "color": "tab:red"},
   #{"type": "fmm",  "n": 10,   "l": 2,    "label": "N10 L2", "marker": ">",  "ls": ":",  "color": "tab:green"},
   #{"type": "fmm",  "n": 10,   "l": 3,    "label": "N10 L3", "marker": "*",  "ls": ":",  "color": "tab:orange"},
]

# 2. Gather the data into the series list
series_list = []
for cfg in configs:
    if cfg["type"] == "cuda":
        path = f"{base_path}_cuda.npz"
    elif cfg["type"] == "cuda_ref":
        path = f"{base_path}_cuda_ref.npz"
    else:
        path = f"{base_path}_fmm_N{cfg['n']}_L{cfg['l']}.npz"
    
    try:
        data = load_hysteresis_data(path)
        cfg["data"] = data
        series_list.append(cfg)
    except FileNotFoundError:
        print(f"Warning: File not found {path}")

# 3. Plot with the high-quality settings
plot_hysteresis_comparison(
    series_list=series_list,
    ms_value=1/mu0,  # Replace with your actual Ms
    fig_width_in=8.2, 
    fig_height_in=3.45 * 1.2,
    font_size=12,
    label_size=14,
    tick_size=10,
    legend_size=14,
    line_width=2,
    marker_size=6.0,
)

In [ ]:
# --- Main Execution ---

# Configuration
base_path = "results/Grid_rasBase_5_nGrains_25_nRef_4_dG_3.75e-09"
N_settings = [6, 6, 10, 10, 17]
L_settings = [2, 3, 2, 3, 2]

# 1. Load Data
#cuda, fmm_data_list = load_comparison_data(base_path, N_settings, L_settings)

# 2. Plot
#fig, ax = plot_comparison(cuda, fmm_data_list, Ms=Ms)

# 3. Save for Poster
# PDF and SVG are vector formats and will not pixelate when scaled for a poster.
save_name = "hysteresis_comparison_vector"
#fig.savefig(f"{save_name}.pdf", format='pdf', bbox_inches='tight')
#fig.savefig(f"{save_name}.svg", format='svg', bbox_inches='tight')
#plt.show()

In [ ]:
# --- Main Execution ---

# Configuration
base_path = "result_09_04/results/Grid_rasBase_5_nGrains_25_nRef_4_dG_3.75e-09"
N_settings = [6]
L_settings = [2]

# 1. Load Data
cuda, fmm_data_list = load_comparison_data(base_path, N_settings, L_settings)

# 2. Plot
fig, ax = plot_comparison(cuda, fmm_data_list, Ms=Ms)

# 3. Save for Poster
# PDF and SVG are vector formats and will not pixelate when scaled for a poster.
save_name = "hysteresis_comparison_vector"
fig.savefig(f"{save_name}.pdf", format='pdf', bbox_inches='tight')
fig.savefig(f"{save_name}.svg", format='svg', bbox_inches='tight')

plt.show()